# Electrical Utility Detection - GPU Training on Colab

This notebook sets up and runs training for detecting electrical utility from satellite images using GPU acceleration.

## Cell 1: Environment Setup

In [1]:
# Check GPU availability
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
    print(
        f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB"
    )

# Install required packages
!pip install segmentation-models-pytorch torchvision pydantic pydantic-settings pyyaml matplotlib

# Optional: Mount Google Drive for data persistence
# from google.colab import drive
# drive.mount('/content/drive')
# %cd /content/drive/MyDrive/  # Change to your project directory

PyTorch version: 2.9.0+cu126
CUDA available: True
GPU: Tesla T4
CUDA version: 12.6
GPU Memory: 15.83 GB
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 7.8 MB/s eta 0:00:00


## Cell 2: Project Setup

In [2]:
import os
import sys
from pathlib import Path

# Set up project paths
project_root = Path.cwd()
print(f"Project root: {project_root}")

# Add src to Python path
src_path = project_root / "src"
if src_path.exists():
    sys.path.insert(0, str(src_path))
    print(f"Added {src_path} to Python path")

# Verify project structure
required_dirs = [
    "src/detect_electrical_utility_from_satellite_images",
    "output_imgs/patches4",
]
for dir_path in required_dirs:
    if (project_root / dir_path).exists():
        print(f"✓ {dir_path} exists")
    else:
        print(f"✗ {dir_path} missing")

# Check for data patches
img_patches = list((project_root / "output_imgs/patches4/img_patches").glob("*.png"))
mask_patches = list((project_root / "output_imgs/patches4/mask_patches").glob("*.png"))
print(f"Found {len(img_patches)} image patches and {len(mask_patches)} mask patches")

Project root: /content
✗ src/detect_electrical_utility_from_satellite_images missing
✗ output_imgs/patches4 missing
Found 0 image patches and 0 mask patches


## Cell 3: Configuration for GPU

In [ ]:
import yaml

# Create GPU-optimized configuration
gpu_config = {
    "preprocessing": {
        "patch_size": 1024,
        "background_fraction": 0.1,
    },
    "paths": {
        "output_dir": "output_imgs/patches4",
        "data_dir": "data",
        "logging_dir_name": "logs/colab_gpu",
    },
    "logging": {
        "logger_lvl": "info",
        "console_handler_lvl": "info",
        "file_handler_lvl": "debug",
    },
    "model": {
        "architecture": "unet",
        "encoder_name": "resnet18",
        "encoder_weights": "imagenet",
        "in_channels": 3,
        "classes": 5,
        "learning_rate": 0.001,
        "batch_size": 8,  # Increased for GPU
        "epochs": 20,  # More epochs for better convergence
        "device": "cuda",
    },
}

# Save configuration
config_path = project_root / "config_colab_gpu.yaml"
with open(config_path, "w") as f:
    yaml.dump(gpu_config, f, default_flow_style=False)

print(f"GPU configuration saved to: {config_path}")
print("\nConfiguration summary:")
print(f"  • Device: {gpu_config['model']['device']}")
print(f"  • Batch size: {gpu_config['model']['batch_size']}")
print(f"  • Epochs: {gpu_config['model']['epochs']}")
print(
    f"  • Architecture: {gpu_config['model']['architecture']} with {gpu_config['model']['encoder_name']}"
)

## Cell 4: Training Execution

In [ ]:
import logging
import time
from pathlib import Path

# Import project modules
from detect_electrical_utility_from_satellite_images.config import AppConfig
from detect_electrical_utility_from_satellite_images.train import train_model
from detect_electrical_utility_from_satellite_images.utils.logging_config import (
    setup_logger,
)

# Load configuration
config_path = project_root / "config_colab_gpu.yaml"
with open(config_path) as f:
    cfg_dict = yaml.safe_load(f)

try:
    cfg = AppConfig(**cfg_dict)
    print("✓ Configuration loaded successfully")
except Exception as e:
    print(f"✗ Configuration error: {e}")
    raise

# Setup logging
setup_logger(
    cfg.paths.logging_dir_name,
    cfg.logging.logger_lvl,
    cfg.logging.console_handler_lvl,
    cfg.logging.file_handler_lvl,
)

# Start training
print("\n" + "=" * 60)
print("STARTING GPU TRAINING")
print("=" * 60)

start_time = time.time()
try:
    train_model(cfg, use_mock_data=False)
    print("\n✓ Training completed successfully!")
except Exception as e:
    print(f"\n✗ Training failed: {e}")
    import traceback

    traceback.print_exc()

end_time = time.time()
print(f"\nTotal training time: {end_time - start_time:.2f} seconds")

# List checkpoints
checkpoint_dir = cfg.paths.output_dir / "checkpoints"
if checkpoint_dir.exists():
    checkpoints = list(checkpoint_dir.glob("*.pt"))
    print(f"\nSaved checkpoints ({len(checkpoints)}):")
    for cp in sorted(checkpoints):
        print(f"  • {cp.name}")

## Cell 5: Evaluation & Visualization

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image

# Load the best model
checkpoint_dir = cfg.paths.output_dir / "checkpoints"
checkpoints = list(checkpoint_dir.glob("best_model_*.pt"))

if checkpoints:
    # Get the latest best model
    latest_checkpoint = sorted(checkpoints)[-1]
    print(f"Loading model from: {latest_checkpoint.name}")

    # Recreate model
    from detect_electrical_utility_from_satellite_images.model import (
        create_model,
        load_checkpoint,
    )

    model = create_model(
        architecture=cfg.model.architecture,
        encoder_name=cfg.model.encoder_name,
        encoder_weights=cfg.model.encoder_weights,
        in_channels=cfg.model.in_channels,
        classes=cfg.model.classes,
        device=cfg.model.device,
    )

    # Load checkpoint
    epoch, loss = load_checkpoint(model, None, latest_checkpoint)
    print(f"Model loaded (epoch {epoch}, loss {loss:.4f})")

    # Set to evaluation mode
    model.eval()

    # Load a sample image for visualization
    img_patches = list((cfg.paths.output_dir / "img_patches").glob("*.png"))
    mask_patches = list((cfg.paths.output_dir / "mask_patches").glob("*.png"))

    if img_patches and mask_patches:
        sample_idx = 0
        img_path = img_patches[sample_idx]
        mask_path = mask_patches[sample_idx]

        # Load and preprocess image
        img = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")

        # Convert to tensors
        from torchvision import transforms

        transform = transforms.Compose(
            [
                transforms.ToTensor(),
            ]
        )

        img_tensor = transform(img).unsqueeze(0).to(cfg.model.device)
        mask_tensor = torch.from_numpy(np.array(mask)).unsqueeze(0).unsqueeze(0)

        # Run inference
        with torch.no_grad():
            output = model(img_tensor)
            prediction = torch.argmax(output, dim=1).cpu().squeeze()

        # Visualize results
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))

        axes[0].imshow(img)
        axes[0].set_title("Input Image")
        axes[0].axis("off")

        axes[1].imshow(mask, cmap="tab20")
        axes[1].set_title("Ground Truth Mask")
        axes[1].axis("off")

        axes[2].imshow(prediction, cmap="tab20")
        axes[2].set_title("Model Prediction")
        axes[2].axis("off")

        plt.tight_layout()
        plt.show()

        # Calculate accuracy
        correct = (prediction == mask_tensor.squeeze()).sum().item()
        total = prediction.numel()
        accuracy = correct / total * 100
        print(f"Sample accuracy: {accuracy:.2f}% ({correct}/{total} pixels)")
    else:
        print("No patches found for visualization")
else:
    print("No checkpoints found for evaluation")

## Next Steps

1. **Run all cells sequentially** - Start from Cell 1
2. **Monitor training progress** - Check logs in `logs/colab_gpu/`
3. **Adjust hyperparameters** if needed:
   - Increase/decrease batch size based on GPU memory
   - Adjust learning rate for better convergence
   - Try different architectures (DeepLabV3, FPN)
4. **Save results** - Download checkpoints or upload to Google Drive

**Note:** If you encounter memory issues, reduce batch size or patch size.